<a id="encrypted-machine-learning-matrix-operations"></a>
# Encrypted Machine Learning: Matrix Operations

This tutorial covers `src/concrete_fhe_toolkit/ml/matrix.py`. Matrix and vector operations are the backbone of most machine learning algorithms. This module provides efficient FHE-compatible primitives for these operations.

<a id="vector-operations-dot-product"></a>
## Vector Operations (Dot Product)

The fundamental operation for linear layers and convolutions.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.ml.matrix import dot_product

def test_dot(x1: int, y1: int, x2: int, y2: int):
    return dot_product([x1, y1], [x2, y2])

compiler = fhe.Compiler(test_dot, {"x1": "encrypted", "y1": "encrypted", "x2": "encrypted", "y2": "encrypted"})
circuit = compiler.compile([(1, 2, 3, 4)])

# [2, -1] dot [4, 5] = (2*4) + (-1*5) = 8 - 5 = 3
result = circuit.encrypt_run_decrypt(2, -1, 4, 5)
assert result == 3
print("✅ Encrypted Dot Product passed!")

<a id="matrix-vector-multiplication"></a>
## Matrix Vector Multiplication

Often used to evaluate a linear layer: `output = Weight_Matrix * input_vector`.

In [ ]:
from concrete_fhe_toolkit.ml.matrix import matrix_vector_multiply

def test_mat_vec(v1: int, v2: int):
    # Public 2x2 matrix
    matrix = [[1, 2], [3, 4]]
    return matrix_vector_multiply(matrix, [v1, v2])

compiler = fhe.Compiler(test_mat_vec, {"v1": "encrypted", "v2": "encrypted"})
circuit = compiler.compile([(1, 1)])

# [1 2]   [5]   [1*5 + 2*10]   [ 25]
# [3 4] * [10] = [3*5 + 4*10] = [ 55]
result = circuit.encrypt_run_decrypt(5, 10)

assert result == [25, 55]
print("✅ Encrypted Matrix-Vector Multiplication passed!")

<a id="matrix-exponentiation"></a>
## Matrix Exponentiation

Calculates the power of a square matrix using the efficient Square-and-Multiply algorithm to minimize multiplicative depth.

In [ ]:
from concrete_fhe_toolkit.ml.matrix import matrix_exp

def test_mat_exp(m11: int, m12: int, m21: int, m22: int):
    matrix = [[m11, m12], [m21, m22]]
    return matrix_exp(matrix, exponent=2) # Matrix squared

compiler = fhe.Compiler(test_mat_exp, {
    "m11": "encrypted", "m12": "encrypted", 
    "m21": "encrypted", "m22": "encrypted"
})

# Input bounds need to be small enough so that M^2 doesn't overflow
circuit = compiler.compile([(1, 1, 1, 1)])

# Matrix M:
# [1 2]
# [3 4]
# M^2:
# [1*1+2*3, 1*2+2*4] = [7,  10]
# [3*1+4*3, 3*2+4*4] = [15, 22]
result = circuit.encrypt_run_decrypt(1, 2, 3, 4)

assert result == [[7, 10], [15, 22]]
print("✅ Encrypted Matrix Exponentiation passed!")